# Исследование надежности заемщиков.

Заказчик — кредитный отдел банка. Нужно разобраться, влияет ли семейное положение и количество детей клиента на факт погашения кредита в срок. Входные данные от банка — статистика о платёжеспособности клиентов.

**Цель проекта**

    1. Выяснить есть ли зависимость между количеством детей и возвратом кредита в срок?
    2. Выяснить есть ли зависимость между семейным положением и возвратом кредита в срок?
    3. Выяснить есть ли зависимость между уровнем дохода и возвратом кредита в срок?
    4. Как разные цели кредита влияют на его возврат в срок?


**Ход исследования**

    1. Обзор данных
    2. Предопработка данных
    3. Вывод по исследованию

### Шаг 1. Обзор данных


Описание данных

children — количество детей в семье

days_employed — общий трудовой стаж в днях

dob_years — возраст клиента в годах

education — уровень образования клиента

education_id — идентификатор уровня образования

family_status — семейное положение

family_status_id — идентификатор семейного положения

gender — пол клиента

income_type — тип занятости

debt — имел ли задолженность по возврату кредитов

total_income — ежемесячный доход

purpose — цель получения кредита

In [3]:
import pandas as pd

data = pd.read_csv('data.csv')

#вывод первых 20-ти строк таблинцы
display(data.head(20))

#Получение общей информации о таблице
data.info()

ModuleNotFoundError: No module named 'pandas'

**Вывод по обзору данных**

Датафрейм содержит 11 столбцов, сразу можно заметить, что в столбце `days_employed` и `total_income` встречаются пропуски. Количество пропусков в этих колонках одинаково и предварительно можно заметить зависимость пропущенных значений в этих колонках (отсутвие рабочего стажа - отсутвие ежемесячного дохода). Возможно получится заменить пропущенные данные медианными значениями, но предварительно необходимо обработать другие ошибки в таблице.

По мимо пропуском можно заметить, что в колонке `education` некоторые значения написаны капсом или с большой буквы - необходимо привести к единообразию. Также в графе `days_employed` присутсвуют аномалии - отрицательные значения, или огромные значения (например 340266.072047 - это 932 года рабочего стажа, не похоже на правду)

### Шаг 1.1 Проверим предположение о зависимости `days_employed` и `total_income`

In [265]:
data[(data['days_employed'].isnull() == True) & (data['total_income'].isnull() == True)].info()

<class 'pandas.core.frame.DataFrame'>
Index: 2174 entries, 12 to 21510
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   children          2174 non-null   int64  
 1   days_employed     0 non-null      float64
 2   dob_years         2174 non-null   int64  
 3   education         2174 non-null   object 
 4   education_id      2174 non-null   int64  
 5   family_status     2174 non-null   object 
 6   family_status_id  2174 non-null   int64  
 7   gender            2174 non-null   object 
 8   income_type       2174 non-null   object 
 9   debt              2174 non-null   int64  
 10  total_income      0 non-null      float64
 11  purpose           2174 non-null   object 
dtypes: float64(2), int64(5), object(5)
memory usage: 220.8+ KB


**Предположение подтвердилось** - в строках, где отсутствуют данные в столбце 'days_employed', отсутствуют данные и по 'salary'.

Также проверим, из каких трудовых групп эти клиенты.

In [266]:
data[(data['days_employed'].isnull() == True) & (data['total_income'].isnull() == True)]['income_type'].value_counts()

income_type
сотрудник          1105
компаньон           508
пенсионер           413
госслужащий         147
предприниматель       1
Name: count, dtype: int64

Трудовые группы разные, заполнить пропуски медианным значением по все таблицы будет не корректно. Будем заполнять пропуски по медианному значению профессии.

### Шаг 1.2 Приведем столбец `education` к единому виду

In [267]:
data['education'] = data['education'].str.lower()

Теперь можем приступать к заполнению пропусков

### Шаг 2.1 Заполнение пропусков `total_income`

In [268]:
#Посчитаем медиану и заполним пропуски медианными значениями с помощью группировки по профессиям

data['total_income'] = data['total_income'].fillna(data.groupby('income_type')['total_income'].transform('median'))

#Проверим датафрейм на наличие пропущенных значений по стобцу total_income

data.info()

#Пропусков больше нет, теперь можем приступить к заполнению пропусков в days_employed

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21525 entries, 0 to 21524
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   children          21525 non-null  int64  
 1   days_employed     19351 non-null  float64
 2   dob_years         21525 non-null  int64  
 3   education         21525 non-null  object 
 4   education_id      21525 non-null  int64  
 5   family_status     21525 non-null  object 
 6   family_status_id  21525 non-null  int64  
 7   gender            21525 non-null  object 
 8   income_type       21525 non-null  object 
 9   debt              21525 non-null  int64  
 10  total_income      21525 non-null  float64
 11  purpose           21525 non-null  object 
dtypes: float64(2), int64(5), object(5)
memory usage: 2.0+ MB


В запонение пропусков в `days_employed` все немного сложнеее, так как трудовой стаж зависит от возраста клиента, медианным значеним по каждой професии заполнить не получится.

**Предлагаю следующий вариант решение:**
    
    1. Заменить все отрицательные значения `days_employed` на положительные
    2. Посчитатать коэфициент рабочего стажа каждого клиента к его возрасту
    3. Заполнить пропуски произведением медианного значения коэффициента по каждой группе на возраст

### Шаг 2.2 Заполнение пропусков `days_employed`

In [269]:
data['days_employed'] = data['days_employed'].abs()

При расчете коэфициента рабочего стажа будем учитывать, что официальное трудоустройство начинается с 16 лет

Добавим доболнительную колонку для коэффициента рабочего стажа

In [270]:
data['work_experience_coefficient'] = data['days_employed'] / ((data['dob_years']-16) * 365)

In [271]:
# Находим медиану
filtered_median = (
    data.groupby('income_type')['work_experience_coefficient'].transform('median')
)

# Заполняем пропуски, используя выравнивание по индексам
data['days_employed'] = data['days_employed'].fillna(
    filtered_median * ((data['dob_years'] - 16) * 365)
)

data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21525 entries, 0 to 21524
Data columns (total 13 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   children                     21525 non-null  int64  
 1   days_employed                21525 non-null  float64
 2   dob_years                    21525 non-null  int64  
 3   education                    21525 non-null  object 
 4   education_id                 21525 non-null  int64  
 5   family_status                21525 non-null  object 
 6   family_status_id             21525 non-null  int64  
 7   gender                       21525 non-null  object 
 8   income_type                  21525 non-null  object 
 9   debt                         21525 non-null  int64  
 10  total_income                 21525 non-null  float64
 11  purpose                      21525 non-null  object 
 12  work_experience_coefficient  19351 non-null  float64
dtypes: float64(3), i

Удаляем вспомогательный столбец и выводим общую информацию о таблице

In [272]:
data = data.drop('work_experience_coefficient', axis=1)
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21525 entries, 0 to 21524
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   children          21525 non-null  int64  
 1   days_employed     21525 non-null  float64
 2   dob_years         21525 non-null  int64  
 3   education         21525 non-null  object 
 4   education_id      21525 non-null  int64  
 5   family_status     21525 non-null  object 
 6   family_status_id  21525 non-null  int64  
 7   gender            21525 non-null  object 
 8   income_type       21525 non-null  object 
 9   debt              21525 non-null  int64  
 10  total_income      21525 non-null  float64
 11  purpose           21525 non-null  object 
dtypes: float64(2), int64(5), object(5)
memory usage: 2.0+ MB


### Шаг 2.3. Изменение типов данных.

Ежемесяцный доход `total_income` и стаж `days_employed` переводим в формат **int**

Так как эти колонки имеют формат **float**, можно спокойно использовать метод `astype`

In [273]:
data['total_income'] = data['total_income'].astype('int64')
data['days_employed'] = data['days_employed'].astype('int64')

data.dtypes

children             int64
days_employed        int64
dob_years            int64
education           object
education_id         int64
family_status       object
family_status_id     int64
gender              object
income_type         object
debt                 int64
total_income         int64
purpose             object
dtype: object

Теперь все данные представлены в удобном формате

### Шаг 2.4. Удаление дубликатов.

Посчитаем количество дубликатов датафрейма и посмотрим на них:

In [274]:
data.duplicated().sum()

71

In [275]:
data[data.duplicated()].sort_values(by=['days_employed', 'total_income']).tail(20)

,children,days_employed,dob_years,education,education_id,family_status,family_status_id,gender,income_type,debt,total_income,purpose
5557,0,350551,58,среднее,1,гражданский брак,1,F,пенсионер,0,118514,сыграть свадьбу
8583,0,350551,58,высшее,0,Не женат / не замужем,4,F,пенсионер,0,118514,дополнительное образование
18755,0,350551,58,среднее,1,женат / замужем,0,F,пенсионер,0,118514,заняться образованием
12736,0,358897,59,среднее,1,женат / замужем,0,F,пенсионер,0,118514,заняться образованием
4851,0,367244,60,среднее,1,гражданский брак,1,F,пенсионер,0,118514,свадьба
21032,0,367244,60,среднее,1,женат / замужем,0,F,пенсионер,0,118514,заняться образованием
19688,0,375590,61,среднее,1,женат / замужем,0,F,пенсионер,0,118514,операции с недвижимостью
9855,0,383936,62,среднее,1,женат / замужем,0,F,пенсионер,0,118514,получение дополнительного образования
10462,0,383936,62,среднее,1,женат / замужем,0,F,пенсионер,0,118514,покупка коммерческой недвижимости
10864,0,383936,62,среднее,1,женат / замужем,0,F,пенсионер,0,118514,ремонт жилью


После просмотра дубликатов, можно сказать, что ничего подозрительного в них нет, выглядит как обычное задвоение информации. Спокойно удаляем дубликаты.

In [276]:
data = data.drop_duplicates()

In [277]:
data.duplicated().sum()

0

### Шаг 2.5. Категоризация целей кредита

В колонке `purpose` хранится информация о целях получения кредита, можем обратить внимание что одни и те же цени указаны разными формулировками. Необходимо проверить количество уникальных целей и привести все к единому виду категорий.

In [278]:
data.purpose.value_counts()

purpose
свадьба                                   791
на проведение свадьбы                     768
сыграть свадьбу                           765
операции с недвижимостью                  675
покупка коммерческой недвижимости         661
операции с жильем                         652
покупка жилья для сдачи                   651
операции с коммерческой недвижимостью     650
покупка жилья                             646
жилье                                     646
покупка жилья для семьи                   638
строительство собственной недвижимости    635
недвижимость                              633
операции со своей недвижимостью           627
строительство жилой недвижимости          624
покупка недвижимости                      621
покупка своего жилья                      620
строительство недвижимости                619
ремонт жилью                              607
покупка жилой недвижимости                606
на покупку своего автомобиля              505
заняться высшим образовани

Просмотрев все цели кредита, я бы их разделил на 4 категорий: **свадьба**, **недвижимость**, **образование**, **автомобиль**.

Добавим стобец `credit_category` и разделим цели кредитов по ранее определенным, 5-ти категория кредитов.

In [279]:
def classify_purpose(purpose):
    purpose = str(purpose)
    if 'свад' in purpose:
        return 'свадьба'
    if 'недв' in purpose or 'жиль' in purpose:
        return 'недвижимость'
    if 'образ' in purpose:
        return 'образование'
    if 'авто' in purpose:
        return 'автомобиль'
    return 'другое'

# Применяем функцию ко всей колонке
data['credit_category'] = data['purpose'].apply(classify_purpose)


#Проверяем получившийся столбец
data.credit_category.value_counts()



credit_category
недвижимость    10811
автомобиль       4306
образование      4013
свадьба          2324
Name: count, dtype: int64

In [280]:
#проверяем полученный датасет

data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 21454 entries, 0 to 21524
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   children          21454 non-null  int64 
 1   days_employed     21454 non-null  int64 
 2   dob_years         21454 non-null  int64 
 3   education         21454 non-null  object
 4   education_id      21454 non-null  int64 
 5   family_status     21454 non-null  object
 6   family_status_id  21454 non-null  int64 
 7   gender            21454 non-null  object
 8   income_type       21454 non-null  object
 9   debt              21454 non-null  int64 
 10  total_income      21454 non-null  int64 
 11  purpose           21454 non-null  object
 12  credit_category   21454 non-null  object
dtypes: int64(7), object(6)
memory usage: 2.3+ MB


Количество значений в колонке соответсвует остальной таблице, категории определились верно

### Шаг 2.6. Категоризация дохода.

Для визуального и смыслового упрощения датафрейма, разделим доход клиентов на категории. 

Обозначим категории по такому диапазону:

    0–30000 — E;

    30001–50000 — D;

    50001–200000 — C;

    200001–1000000 — B;

    1000001 и выше — A.

In [281]:
def classify_total_category(total):
    if total <= 30000:
        return 'E'
    elif 30001 <= total <= 50000:
        return 'D'
    elif 50001 <= total <= 200000:
        return 'C'
    elif 200001 <= total <= 1000000:
        return 'B'
    elif total > 1000001:
        return 'A'

data['total_income_category'] = data['total_income'].apply(classify_total_category)

data['total_income_category'].value_counts()

total_income_category
C    16015
B     5042
D      350
A       25
E       22
Name: count, dtype: int64

### Шаг 2.7 Категоризация количества детей

Категоризуем клиентов по количеству детей. Разделим их по такому признаку:

    0 детей - "нет детей"
    1-2 ребенка - "1-2 ребенка"
    3 и более детей - "многодетные"

In [282]:
data

,children,days_employed,dob_years,education,education_id,family_status,family_status_id,gender,income_type,debt,total_income,purpose,credit_category,total_income_category
0,1,8437,42,высшее,0,женат / замужем,0,F,сотрудник,0,253875,покупка жилья,недвижимость,B
1,1,4024,36,среднее,1,женат / замужем,0,F,сотрудник,0,112080,приобретение автомобиля,автомобиль,C
2,0,5623,33,среднее,1,женат / замужем,0,M,сотрудник,0,145885,покупка жилья,недвижимость,C
3,3,4124,32,среднее,1,женат / замужем,0,M,сотрудник,0,267628,дополнительное образование,образование,B
4,0,340266,53,среднее,1,гражданский брак,1,F,пенсионер,0,158616,сыграть свадьбу,свадьба,C
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21520,1,4529,43,среднее,1,гражданский брак,1,F,компаньон,0,224791,операции с жильем,недвижимость,B
21521,0,343937,67,среднее,1,женат / замужем,0,F,пенсионер,0,155999,сделка с автомобилем,автомобиль,C
21522,1,2113,38,среднее,1,гражданский брак,1,M,сотрудник,1,89672,недвижимость,недвижимость,C
21523,3,3112,38,среднее,1,женат / замужем,0,M,сотрудник,1,244093,на покупку своего автомобиля,автомобиль,B


In [283]:
def children_cat(children):
    if  children == 0:
        return 'нет детей'
    elif 1 <= children <= 2:
        return '1-2 детей'
    else:
        return 'многодетная'
data['children_category'] = data['children'].apply(children_cat)

#Проверяем данные

data.info()

#Пропусков нет, можно смело переходить на следующий этап

<class 'pandas.core.frame.DataFrame'>
Index: 21454 entries, 0 to 21524
Data columns (total 15 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   children               21454 non-null  int64 
 1   days_employed          21454 non-null  int64 
 2   dob_years              21454 non-null  int64 
 3   education              21454 non-null  object
 4   education_id           21454 non-null  int64 
 5   family_status          21454 non-null  object
 6   family_status_id       21454 non-null  int64 
 7   gender                 21454 non-null  object
 8   income_type            21454 non-null  object
 9   debt                   21454 non-null  int64 
 10  total_income           21454 non-null  int64 
 11  purpose                21454 non-null  object
 12  credit_category        21454 non-null  object
 13  total_income_category  21454 non-null  object
 14  children_category      21454 non-null  object
dtypes: int64(7), object(8)
m

Также перед переходом к анализу данных я бы удалил все колонки, на основе которых мы категоризировали данные

In [284]:
data = data.drop('purpose', axis=1)
data = data.drop('total_income', axis=1)
data = data.drop('children', axis=1)


## Шаг 3. Анализ полученных данных

Создадим функцию для выявления зависимости:

In [285]:
def relation(category):
    return data.groupby(category)['debt'].mean().to_frame().sort_values(by='debt')

**1. Выяснить есть ли зависимость между количеством детей и возвратом кредита в срок?**

In [286]:
relation('children_category')

,debt
children_category,
нет детей,0.075438
многодетная,0.079523
1-2 детей,0.093003


### Вывод

Клиенты, у которых остутсвуют дети, менее склоны к задолжностям по кредиту, но и многодетные семьи почти так же исправно выплачивают займ как и клиенты без детей. По полученным данным можно сказать, что неиболее склонные к просрочкам, являются семьи с 1-2 детьми.

**2. Выяснить есть ли зависимость между семейным положением и возвратом кредита в срок?**

In [287]:
relation('family_status')

,debt
family_status,
вдовец / вдова,0.065693
в разводе,0.071130
женат / замужем,0.075452
гражданский брак,0.093471
Не женат / не замужем,0.097509


### Вывод

При ответе на этот вопрос можно сделать вывод, что самые имправные плательщики это вдовцы/вдовы, а самые склонные к неуплате это не женатые/ не замужние и люди состоящие в гражданском браке

**3. Выяснить есть ли зависимость между уровнем дохода и возвратом кредита в срок?**

In [288]:
relation('total_income_category')

,debt
total_income_category,
D,0.060000
B,0.070607
A,0.080000
C,0.084920
E,0.090909


Наиболее склонные к просрочкам клиенты, это клиенты с самым низким доходом **(E)**, в целом понятно чем это может быть вызвано. Интересным является то что самые богатые клиенты так же являются не самой добросовестной категорией **(А)**, они занимают центровую позицию среди должников. Самыми успешными в плане выплат являются клиенты с уровнем дохода 30001–50000 **(D)**.

**4. Как разные цели кредита влияют на его возврат в срок?**

In [289]:
relation('credit_category')

,debt
credit_category,
недвижимость,0.072334
свадьба,0.080034
образование,0.092200
автомобиль,0.093590


Заемщики, берущие кредит для приобретения/проведение операций с жильем, наиболее ответственны и менее склонны нарушать обязательства по выплатам кредита в срок.

# Итоги исследования:

1. **Зависимость от количества детей:** Самыми надежными заемщиками являются клиенты **без детей** (просрочка **7.54%**). Появление детей повышает риски: у клиентов с **1-2 детьми** просрочка возрастает до максимальных **9.30%**. Интересно, что **многодетные** клиенты платят чуть лучше (**7.95%**), однако для окончательного вывода по ним стоит проверить размер выборки (их обычно в разы меньше)

2. **Зависимость от семейного положения:** Самый низкий риск невозврата демонстрируют люди, побывавшие в **официальном браке**, но потерявшие партнера **(вдовы/вдовцы — 6.57%)**, а также люди в **разводе (7.11%)** и **женат/замужем (7.45%)**. В зоне высокого риска находятся те, кто не оформил отношения официально: клиенты в **гражданском браке (9.35%)** и **холостые/незамужние (9.75%)**

3. **Зависимость от уровня дохода:** Финансовая дисциплина не имеет строго линейной зависимости от денег. Идеальными плательщиками выступают люди с доходом категории **D (6.00%)**. Хуже всего платят клиенты с самым низким доходом — категория **E (9.09%)**, а также самая массовая категория **C (8.49%)**

4. **Влияние целей кредита:** Операции с **недвижимостью (7.23%)** — самые осознанные и безопасные для банка. Напротив, займы на **образование (9.22%)** и **автомобили (9.36%)** сопряжены с самым высоким уровнем невозврата.

### Рейтинг заемщиков по уровню надежности

В таблице ниже категории сгруппированы и отранжированы от самых надежных (зеленая зона) до самых рискованных (красная зона).

| Уровень надежности | Категория заемщика | Фактор (Критерий) | Доля просрочки (`debt`) | Риск для банка |
| :--- | :--- | :--- | :---: | :--- |
| 🟢 **Высокий**<br>(Топ-надежные) | Категория дохода D<br>Вдовец / вдова<br>В разводе<br>Недвижимость | Доход<br>Семейное положение<br>Семейное положение<br>Цель кредита | 0.0600<br>0.0657<br>0.0711<br>0.0723 | Минимальный.<br>Идеальные кандидаты для одобрения и сниженных ставок. |
| 🟡 **Средний**<br>(Умеренный риск) | Женат / замужем<br>Нет детей<br>Многодетная<br>Свадьба<br>Категория дохода B | Семейное положение<br>Дети<br>Дети<br>Цель кредита<br>Доход | 0.0755<br>0.0754<br>0.0795<br>0.0800<br>0.0706 | Стабильный.<br>Базовый уровень риска. Стандартные условия кредитования. |
| 🟠 **Пониженный**<br>(Внимание) | Категория дохода A<br>Категория дохода C<br>Категория дохода E | Доход<br>Доход<br>Доход | 0.0800<br>0.0849<br>0.0909 | Повышенный.<br>Требуется дополнительная проверка платежеспособности. |
| 🔴 **Низкий**<br>(Максимальный риск) | Образование<br>1-2 детей<br>Гражданский брак<br>Автомобиль<br>Не женат / не замужем | Цель кредита<br>Дети<br>Семейное положение<br>Цель кредита<br>Семейное положение | 0.0922<br>0.0930<br>0.0935<br>0.0936<br>0.0975 | Критический.<br>Рекомендуется залог, поручительство или повышенная ставка. |